In [1]:
import numpy as np
import pandas as pd
import tifffile as tif
from tqdm import tqdm

In [ ]:
# name = "Aq1T0_270z_770z"
# name = "Aq3T1_300z_1050z"
name = "Aq1T1_2000z_2600z"

data = pd.read_csv(f"../data/crops/csv/v1/{name}.csv", sep=";")
x = tif.imread(f"../data/crops/im/{name}.tif")

print(x.shape)
# Original order must be Z,Y,X
x = np.swapaxes(x, -1, 0)
# Now X, Y, Z

print(len(data))
print(x.shape)

(600, 1900, 1900)
2167
(1900, 1900, 600)


In [3]:
# Create groundtruth data
y = np.zeros_like(x)

print(y.shape)
print(x.shape)
radius = 8

def s2i(value):
    if isinstance(value, str):
        value = value.replace(",", ".")
    return int(float(value))

def pt_in_dim(pt, dim):
    return pt >= 0 and pt < dim

def point_in_array(pt, array):
    for i_dim in range(len(array.shape)):
        if not pt_in_dim(pt[i_dim], array.shape[i_dim]):
            print(f"Discard point {pt} since at dim {i_dim} point {pt[i_dim]} not in (0;{array.shape[i_dim]})")
            return False
    return True

# Prepare a sphere template
sphere_array = np.zeros((radius*2, radius*2, radius*2), dtype=np.uint8)
center_x, center_y, center_z = sphere_array.shape[0] // 2, sphere_array.shape[1] // 2, sphere_array.shape[1] // 2
X, Y, Z = np.ogrid[:sphere_array.shape[0], :sphere_array.shape[1], :sphere_array.shape[2]]
mat_distance = (X - center_x) ** 2 + (Y - center_y) ** 2 + (Z - center_z) ** 2
sphere_array[mat_distance <= radius**2] = 255

print(len(data))

def get_slice(pos, array_dim, patch_dim):
    half_patch_size = patch_dim//2
    array_start = max(0, pos - half_patch_size)
    patch_start = half_patch_size - (pos - array_start)
    array_end = min(array_dim - 1, pos + half_patch_size)
    patch_end = array_end - pos + half_patch_size
        
    assert array_start <= array_end, print(f"start: {array_start}, end: {array_end}")
    assert patch_start <= patch_end, print(f"start: {patch_start}, end: {patch_end}")
    
    return array_start, array_end, patch_start, patch_end

skipped = 0
for _, center in tqdm(data.iterrows()):
    center_x, center_y, center_z = s2i(center["Xcoords"]), s2i(center["Ycoords"]), s2i(center["Zcoords"])
    center = center_x, center_y, center_z
    if not point_in_array(center, y):
        skipped += 1
        continue
    

    mat_start_x, mat_end_x, patch_start_x, patch_end_x = get_slice(center_x, y.shape[0], sphere_array.shape[0])
    mat_start_y, mat_end_y, patch_start_y, patch_end_y = get_slice(center_y, y.shape[1], sphere_array.shape[1])
    mat_start_z, mat_end_z, patch_start_z, patch_end_z = get_slice(center_z, y.shape[2], sphere_array.shape[2])

    y[mat_start_x:mat_end_x, mat_start_y:mat_end_y, mat_start_z:mat_end_z] = \
        sphere_array[patch_start_x:patch_end_x, patch_start_y:patch_end_y, patch_start_z:patch_end_z]

if skipped > 0:
    print(f"WARNING: skipped {skipped} points that were not in the matrix")

tif.imwrite(f"../data/crops/label/{name}_truth.tif", y)
tif.imwrite(f"../data/crops/label/{name}_im_xyz.tif", x)




(1900, 1900, 600)
(1900, 1900, 600)
2167


2167it [00:00, 32117.04it/s]
